# 蜃景 × Colab 一条龙（FLUX 出图 / Wan2.2-A14B 出片 / 人物 LoRA）

**怎么用：菜单「代码执行程序 → 全部运行」(Run all)。** 断线 / 被回收后，再点一次 **Run all** 即可——模型、ai-toolkit、pip 依赖全持久化在 Drive，**不重下**，几分钟恢复。

**跑前一次性：**
1. 运行时选 **GPU（H100 / RTX PRO 6000 Blackwell / A100-80G 都行）** —— 第 1 格会**自动探测显卡**并选最优档：H100/Ada/Blackwell（原生 FP8）→ fp8 快档（80G 再加 highvram）+ 关 sage；A100-80G（无原生 FP8）→ fp16 原生档（不开 highvram，避 OOM）；给啥卡都不配错。（代码执行程序 → 更改运行时类型 → 选 GPU；H100/Blackwell 需 Colab Pro+，且不保证每次分到）
2. 右侧 🔑 Secrets 加 **CIVITAI_TOKEN**(出图底模用)。**全程不需要 HF_TOKEN**(ae 走非 gated 的 FLUX.1-schnell)。
3. 第 1 格 `DEEPSEEK_KEY` **可留空**（出图/出片/EP1 不需要 LLM）。

**已内置抗断线：** 所有服务用 `start_new_session` **脱离内核**——你停某个 cell / 中断，**不会**杀掉 ComfyUI / 后端。但仍**别**重启运行时、别让标签长时间断连（免费 Colab 空闲/12h/回收是天性，断了就再 Run all）。

In [ ]:
# === 1. 参数 + Drive + 持久化路径（全部落 Drive）+ GPU 自动探测 ===
REPO_URL = 'https://github.com/aw3n1998/build-with-langchain.git'
BRANCH   = 'main'
DEEPSEEK_KEY = ''        # 出图/出片/EP1 不需要，可留空
# ── 出图底模：CivitAI 的 DAC_Fluxed_Style（FLUX UNET-only，走自带 flux 模板）──
# token 从 Colab Secrets 读(右侧🔑加 CIVITAI_TOKEN)，★绝不写进代码/仓库★。版本号已填好：
CIVITAI_VERSION = '762901'   # 模型版本号(Download 链接里 models/ 后的数字)
CIVITAI_FILEID  = '676281'   # 文件变体(该版本多文件时指定;单变体可留空 '')
try:
    from google.colab import userdata; _CT = userdata.get('CIVITAI_TOKEN')
except Exception:
    _CT = ''
FLUX_BASE_FILE = 'fluxed-up.safetensors'   # 本地存名(随意;COMFYUI_FLUX_UNET 自动跟随)
FLUX_BASE_KIND = 'unet'
FLUX_BASE_REPO = ''
_q = '?token=' + (_CT or '') + (('&fileId=' + CIVITAI_FILEID) if CIVITAI_FILEID else '')
FLUX_BASE_URL  = ('https://civitai.com/api/download/models/' + CIVITAI_VERSION + _q) if (_CT and CIVITAI_VERSION.isdigit()) else ''
if not FLUX_BASE_URL: print('⚠️ 未取到 CIVITAI_TOKEN → 右侧🔑加 Secret CIVITAI_TOKEN 后再 Run all')
# 换别的底模: 改 CIVITAI_VERSION/FILEID;或换 HF UNET-only(设 FLUX_BASE_REPO/FILE, FLUX_BASE_KIND='unet')
import os, shutil
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # optional: add HF_TOKEN in Colab Secrets to speed HF downloads (anonymous also works for public models)
except Exception:
    pass
os.environ.update({'FLUX_BASE_REPO':FLUX_BASE_REPO,'FLUX_BASE_FILE':FLUX_BASE_FILE,
                   'FLUX_BASE_URL':FLUX_BASE_URL,'FLUX_BASE_KIND':FLUX_BASE_KIND})
!nvidia-smi -L

# ── GPU 自动探测：据算力(是否原生FP8)+显存 选 出片精度/显存模式/注意力 ───────────────
# Colab 不保证给你选的卡(常被降级)，所以这里实测而非写死。
# sm>=8.9(Ada/Hopper/Blackwell)=原生FP8 → fp8 又快又省；A100(sm8.0)无 → 走 fp16。
import torch as _torch
if _torch.cuda.is_available():
    _cc = _torch.cuda.get_device_capability(0); _name = _torch.cuda.get_device_name(0)
    _vram = _torch.cuda.get_device_properties(0).total_memory / (1024**3)
    _native_fp8 = (_cc[0] > 8) or (_cc[0] == 8 and _cc[1] >= 9)
    if _native_fp8:
        # 原生FP8(H100/L40S/4090/L4/Blackwell)：fp8 模板；80G 才开 highvram；Wan 上全局 sage 有噪声 bug → 关，用 sdpa
        _i2v, _prec, _hv, _sage = 'comfyui_workflows/i2v_fp8_template.json', 'fp8', ('1' if _vram >= 70 else '0'), '0'
    elif _vram >= 70:
        # A100-80G(无原生FP8)：fp16 原生(fp8 只会模拟更亏)；不开 highvram(避 FLUX↔Wan 叠加 OOM)；sage 照旧
        _i2v, _prec, _hv, _sage = 'comfyui_workflows/i2v_bf16_template.json', 'fp16', '0', '1'
    else:
        # 显存<70G 且无原生FP8(A100-40G/V100)：fp16 双专家放不下 → 回退 fp8(会模拟、较慢)
        _i2v, _prec, _hv, _sage = 'comfyui_workflows/i2v_fp8_template.json', 'fp8', '0', '1'
        print('⚠️ 显存<70G 且无原生FP8 → 回退 fp8(会模拟、较慢)；建议换 80G 卡或降分辨率')
    os.environ.update({'COMFYUI_WORKFLOW_I2V': _i2v, 'I2V_PRECISION': _prec,
                       'MIRAGE_HIGHVRAM': _hv, 'MIRAGE_USE_SAGE': _sage})
    print(f'🖥 GPU: {_name} | sm{_cc[0]}.{_cc[1]} | {_vram:.0f}G | 原生FP8={_native_fp8}'
          f' → 出片 {_prec} | highvram={_hv} | sage={_sage}')
else:
    os.environ.update({'COMFYUI_WORKFLOW_I2V': 'comfyui_workflows/i2v_bf16_template.json',
                       'I2V_PRECISION': 'fp16', 'MIRAGE_HIGHVRAM': '0', 'MIRAGE_USE_SAGE': '1'})
    print('⚠️ 没检测到 GPU！代码执行程序 → 更改运行时类型 → 选 GPU(H100/Blackwell/A100)')

# ── 让脱离内核的后台子进程(ComfyUI/后端)继承内核这套 torch ───────────────────────
# Colab 偶发"内核 torch 新(cu128) ↔ 系统 python torch 旧(cu121)"环境分裂(Blackwell/sm120 上踩过):
# 子进程默认走系统 python 会 import 到旧 torch → ComfyUI "no kernel image"/custom_op 缺失起不来。
# 把内核 sys.path 灌进子进程 PYTHONPATH(restart_service 的 Popen 不传 env→继承 os.environ)，
# 强制 ComfyUI/后端也用内核这套包(含正确的 torch)。对齐好的环境(纯 A100/H100)上是无害冗余。
import sys
os.environ['PYTHONPATH'] = os.pathsep.join(p for p in sys.path if p)
print('PYTHONPATH 已设 → 子进程(ComfyUI/后端)复用内核 torch')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'
CACHE=DRIVE+'/mirage_models'                 # 模型（持久化）
TOOLS=DRIVE+'/mirage_tools'                   # ai-toolkit
PIP_CACHE=TOOLS+'/pip_cache'                  # pip 缓存（依赖不重下）
# 旧名自动迁移（改名 Mirage 前的 agentlab_models）
if os.path.isdir(DRIVE+'/agentlab_models') and not os.path.isdir(CACHE):
    shutil.move(DRIVE+'/agentlab_models', CACHE); print('agentlab_models → mirage_models 迁移完成')
for p in (CACHE, TOOLS, PIP_CACHE): os.makedirs(p, exist_ok=True)
os.environ['PIP_CACHE_DIR']=PIP_CACHE
os.environ['HF_HOME']=TOOLS+'/hf_cache'; os.makedirs(os.environ['HF_HOME'], exist_ok=True)  # HF cache (PuLID EVA-CLIP etc) on Drive
print('环境就绪 | 模型:', CACHE, '| pip缓存:', PIP_CACHE)

In [ ]:
# === 1b. (可选) 启用 LTX-Video 2.3：与 Wan2.2 并列的「快/音视频一体」出片档 ===
# 把 ENABLE_LTX 改 True 再 Run all，就会：
#   ① ComfyUI 升到 v0.16+（Wan 也跟着上新版——向下兼容能跑，但请重验一镜；且需 torch≥2.4）
#   ② setup 装 LTX 官方增强节点  ③ download 下 LTX 22B 权重（+Gemma3 文本编码器）
#   ④ 后端把 ltx2 并列进出片模型下拉（参数卡 LTX 专属：档位/8n+1帧数/32倍数尺寸/guidance）
# False（默认）= 完全不动现有 Wan 链路（ComfyUI 仍钉 v0.3.75）。这些 env 会被后面的 setup/download/后端继承。
# ★首跑前务必按 ComfyUI v0.16+ 自带官方 LTX i2v 模板，核对/替换 comfyui_workflows/ltx_i2v_template.json（脚手架）。
import os
ENABLE_LTX = False
if ENABLE_LTX:
    os.environ.update({
        'COMFY_REF':     os.environ.get('COMFY_REF') or 'v0.16.1',  # LTX 2.3 需 ComfyUI v0.16+（可改更高 tag）
        'SETUP_LTX':     '1',          # setup.sh 装官方 LTX 增强节点（ComfyUI-LTXVideo）
        'DOWNLOAD_LTX':  '1',          # download_models.sh 下 LTX 权重（dev fp8 ~22G）
        'LTX2_ENABLED':  'true',       # 后端把 ltx2 并列进用户模型下拉
        # 双实例(LTX 单开端口/单开机器、与 Wan 旧版隔离)再设：os.environ['COMFYUI_LTX_BASE_URL']='http://127.0.0.1:8189'
    })
    print('🎬 LTX 已启用 → ComfyUI 升级到', os.environ['COMFY_REF'],
          '| 下 LTX 权重 | 出片下拉里会多一个「LTX-Video 2.3」(Wan 仍在)。⚠️ 升级后请先重验 Wan 一镜再用。')
else:
    print('LTX 关(默认)：ComfyUI 钉 v0.3.75，仅 Wan2.2 出片。要 LTX 把本格 ENABLE_LTX 改 True 再 Run all。')

In [ ]:
# === 2. 拉取/更新仓库（Mirage）===
import os, sys
REPO_URL=globals().get('REPO_URL','https://github.com/aw3n1998/build-with-langchain.git')
BRANCH=globals().get('BRANCH','main')
%cd /content
if not os.path.isdir('mirage/.git'):
    !git clone -b {BRANCH} {REPO_URL} mirage
else:
    !cd mirage && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
%cd /content/mirage
sys.path.insert(0, '/content/mirage/colab')   # 之后可 import persist
print('仓库就绪 @', BRANCH)

In [ ]:
# === 3. 装 ComfyUI + 自定义节点（幂等；自愈空壳）===
!bash /content/mirage/colab/setup.sh

In [ ]:
# === 4. 模型 move-safe 软链 Drive（不删数据）+ 下模型（幂等，不重下）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.link_dir(globals().get('TOOLS','/content/drive/MyDrive/mirage_tools')+'/insightface_home', '/root/.insightface')  # insightface antelopev2 also on Drive
CACHE=globals().get('CACHE','/content/drive/MyDrive/mirage_models')
persist.link_models(CACHE, '/content/ComfyUI/models',
                    ['unet','checkpoints','clip','vae','audio_encoders','loras','pulid','text_encoders','diffusion_models','insightface','upscale_models'])
print('models 已 move-safe 软链 Drive')
!bash /content/mirage/colab/download_models.sh

In [ ]:
# === 5. 起 ComfyUI（硬重启：杀旧+起新；脱离内核，停 cell 不杀）===
# 子进程靠第1格设的 PYTHONPATH 继承内核 torch（split-torch/Blackwell 环境必需，否则 "no kernel image" 起不来）。
# ⚠️ ComfyUI 只在启动时扫一次模型目录 → 新下/新增模型后必须重跑本格，否则出图/出片会报模型 "not in []"。
# --highvram / --use-sage-attention 由第1格 GPU 探测结果(os.environ)决定：
#   H100 等原生FP8+80G → 开 highvram(fp8 模型小、常驻不 OOM、步速最快)；Wan 上 sage 有噪声 bug → 关，用 sdpa
#   A100-80G → 不开 highvram(避 FLUX↔Wan 叠加 OOM)；sage 装了就用
import os, sys; sys.path.insert(0, '/content/mirage/colab'); import persist
import importlib.util
_args = ['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188']
if os.environ.get('MIRAGE_HIGHVRAM') == '1':
    _args.append('--highvram')
if os.environ.get('MIRAGE_USE_SAGE', '1') == '1' and importlib.util.find_spec('sageattention'):
    _args.append('--use-sage-attention')
print('ComfyUI 启动参数:', ' '.join(_args[2:]))
persist.restart_service('ComfyUI', _args,
    'http://127.0.0.1:8188/', '/content/comfyui.log', 8188, 'ComfyUI/main.py', timeout=180)

In [ ]:
# === 6. 装后端依赖（pip 走 Drive 缓存）+ 写干净 .env（按底模 kind 配出图 + 按 GPU 配出片）===
%cd /content/mirage
!pip -q install -r requirements.txt
!pip -q install fastembed edge-tts
import re, shutil, os
DEEPSEEK_KEY=globals().get('DEEPSEEK_KEY','')
shutil.copy('.env.colab','.env')
env=open('.env').read()
if DEEPSEEK_KEY: env=re.sub(r'OPENAI_API_KEY=.*','OPENAI_API_KEY='+DEEPSEEK_KEY,env)
_file=globals().get('FLUX_BASE_FILE','flux1-dev.safetensors'); _kind=globals().get('FLUX_BASE_KIND','unet')
def _setenv(e,k,v):
    if (k+'=') in e:
        return re.sub(k+r'=.*', k+'='+v, e)
    return e + ('' if e.endswith(chr(10)) else chr(10)) + k + '=' + v
if _kind=='checkpoint':
    env=_setenv(env,'COMFYUI_FLUX_CKPT',_file)
    env=_setenv(env,'COMFYUI_WORKFLOW_T2I','comfyui_workflows/t2i_checkpoint_template.json')
else:
    env=_setenv(env,'COMFYUI_FLUX_UNET',_file)
    env=_setenv(env,'COMFYUI_WORKFLOW_T2I','')
# 出片 i2v 模板按第1格 GPU 探测结果写(H100→fp8 / A100→fp16)，与下载档一致
_i2v=os.environ.get('COMFYUI_WORKFLOW_I2V','comfyui_workflows/i2v_bf16_template.json')
env=_setenv(env,'COMFYUI_WORKFLOW_I2V',_i2v)
open('.env','w').write(env)
print('.env 写好（出图底模:'+_file+' kind='+_kind+' / 出片:'+os.environ.get('I2V_PRECISION','fp16')+' '+os.path.basename(_i2v)+' / SSH 关 / LLM key 可空）')

In [ ]:
# === 7. (可选) 构建前端 → 单端口出 UI。不需要可跳过，直接用 API / 第 11 格全自动 ===
%cd /content/mirage/frontend
!npm install --silent && npm run build || echo '前端构建跳过（用 API 即可）'
%cd /content/mirage

In [ ]:
# === 8. 起后端（硬重启：杀旧+起新，确保读到第6格刚写的 .env，不被旧后端霸占）===
import sys; sys.path.insert(0, '/content/mirage/colab'); import persist
persist.restart_service('后端',
    ['uvicorn','mirage.main_api:app','--host','0.0.0.0','--port','8000'],
    'http://127.0.0.1:8000/api/health', '/content/api.log', 8000, 'uvicorn', cwd='/content/mirage', timeout=120)

In [ ]:
# === 9. 灌入《ASHBORN》EP1（项目+4角色+8分镜；对口型 2/5/8）===
# 工作目录留本地（SQLite 不能放 Drive FUSE，会锁死）；成片在第 11 格末尾另拷 Drive。
import os; WS='/content/cael_video_out'; os.makedirs(WS, exist_ok=True)
!python /content/mirage/scripts/seed_ashborn_ep1.py {WS}

In [ ]:
# === 10. cloudflared 暴露后端（脱离内核；轮询到公网地址即刻打印，秒级见 URL）===
import sys, re, time, os, subprocess
sys.path.insert(0, '/content/mirage/colab'); import persist
# 1) 先杀掉上次还活着的隧道——它占着 /usr/local/bin/cloudflared，否则覆盖时报 Text file busy
subprocess.run('pkill -9 -f cloudflared 2>/dev/null; sleep 2', shell=True)
# 2) 二进制不在才下载（已存在不覆盖；正在运行也不会再撞 Text file busy）
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/'
                   'cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared',
                   shell=True)
open('/content/cf.log', 'w').close()                       # 清旧日志，免读到上一次的死地址
persist.start_bg(['cloudflared', 'tunnel', '--url', 'http://localhost:8000'], '/content/cf.log')
# 3) 轮询：地址一出现就立刻打印（通常几秒），不再盲等 12s、不再"重跑本格"
url = None
for _ in range(60):                                        # 最多等 60s，出来即停
    time.sleep(1)
    m = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cf.log').read())
    if m:
        url = m[-1]; break
if url:
    print('✅ 公网地址:', url)
else:
    print('⚠ 隧道 60s 未就绪 → !tail -20 /content/cf.log（多半 trycloudflare 限流，重跑本格即可）')
print('打开它 → 工作目录选 /content/cael_video_out → 选 ASHBORN → 一键出图→选图→出片合成')

In [ ]:
# === 11. 全自动出整集：出图 → 自动选第1张 → 出片 → 合成。成片末尾拷 Drive ===
# 两档:DRAFT=True → 出片走【极速档(Lightning 4步)打样】,先快速过一遍给你在网页预览挑;
#       满意后用 11b 把【选中的镜号】按满档【精修重渲】。DRAFT=False → 直接满档精修出全片(慢)。
import requests, time, glob, shutil, os
API='http://127.0.0.1:8000/api'; WS='/content/cael_video_out'
DRAFT = True                                  # True=极速打样 / False=直接精修全片
proj=lambda pid: requests.get(f"{API}/pipeline/project/{pid}",params={"workspace":WS},timeout=90).json()
pid=requests.get(f"{API}/pipeline/projects",params={"workspace":WS}).json()["projects"][0]["project_id"]
print("项目:", pid, "| 出片档:", "极速打样(Lightning 4步)" if DRAFT else "精修(满档)")
requests.post(f"{API}/pipeline/batch_generate",json={"project_id":pid,"workspace":WS},timeout=30)
t0=time.time()
while True:
    scs=proj(pid).get("scenes",[]); have=sum(1 for s in scs if s["candidates"])
    print(f"  出图 {have}/{len(scs)} ({int(time.time()-t0)}s)")
    if scs and have>=len(scs): break
    if time.time()-t0>3600: print("出图超时，看 comfyui.log"); break
    time.sleep(20)
for s in proj(pid)["scenes"]:
    if s["candidates"] and not s["selected"]:
        requests.post(f"{API}/pipeline/select",json={"scene_id":s["scene_id"],"asset_id":s["candidates"][0]["assetId"],"workspace":WS},timeout=30)
print("  已选图，"+("极速打样出片" if DRAFT else "精修出片")+"+合成…")
requests.post(f"{API}/pipeline/batch_finish",json={"project_id":pid,"workspace":WS,"lightning":DRAFT},timeout=30)
t0=time.time()
while True:
    try: p=proj(pid)
    except Exception as e: print("  查询失败重试:",e); time.sleep(15); continue
    scs=p.get("scenes",[]); done=sum(1 for s in scs if s.get("video"))
    print(f"  出片 {done}/{len(scs)} 整集:{'OK' if p.get('episode') else '合成中...'} ({int(time.time()-t0)}s)")
    if p.get("episode"):
        print("整集成片:", p["episode"]["url"])
        eps=glob.glob(WS+'/**/*.mp4', recursive=True)
        if eps:
            dst='/content/drive/MyDrive/mirage_episodes'; os.makedirs(dst, exist_ok=True)
            for f in eps: shutil.copy(f, dst)
            print(f'已把 {len(eps)} 个 mp4 拷到 Drive(持久化): {dst}')
        if DRAFT: print('👉 这是极速草稿。去 cell10 公网地址预览 → 记下要精修的镜号 → 跑【11b】重渲。')
        break
    if time.time()-t0>6000: print("出片超时，看 comfyui.log"); break
    time.sleep(30)

In [ ]:
# === 11b. 精修重渲「选中的镜号」（极速打样满意后，把要的几镜按满档重渲 + 重合成整集）===
# 用法：cell11 极速打样 → cell10 公网地址预览草稿 → 把要精修的镜号填进 FINAL_SCENES → 跑本格。
#   FINAL_SCENES=[] 表示精修全部已出片的镜。H100 走原生 fp8 满档；A100 走 fp16 满档。
#   只重渲你选中的镜（其余草稿保留）→ 算力都花在留用的镜上，利用率最高。
import requests, time
API='http://127.0.0.1:8000/api'; WS='/content/cael_video_out'
FINAL_SCENES = []          # 例：[1,3,5]；留空=全部重渲精修
pid=requests.get(f"{API}/pipeline/projects",params={"workspace":WS}).json()["projects"][0]["project_id"]
scs=requests.get(f"{API}/pipeline/project/{pid}",params={"workspace":WS},timeout=90).json().get("scenes",[])
ids=[s["scene_id"] for s in scs if s.get("video") and (not FINAL_SCENES or s["scene_number"] in FINAL_SCENES)]
if not ids:
    print("没有可精修的镜（先跑 cell11 极速打样出片）。")
else:
    print(f"精修重渲 {len(ids)} 镜（lightning=关·满档）+ 重合成整集…")
    requests.post(f"{API}/pipeline/batch_finish",
                  json={"project_id":pid,"workspace":WS,"lightning":False,"scene_ids":ids},timeout=30)
    t0=time.time()
    while True:
        p=requests.get(f"{API}/pipeline/project/{pid}",params={"workspace":WS},timeout=90).json()
        done=sum(1 for s in p.get("scenes",[]) if s.get("video"))
        print(f"  精修 {done}/{len(p.get('scenes',[]))} 整集:{'OK' if p.get('episode') else '合成中...'} ({int(time.time()-t0)}s)")
        if p.get("episode"): print("✅ 精修整集:", p["episode"]["url"]); break
        if time.time()-t0>9000: print("超时，看 comfyui.log"); break
        time.sleep(30)

In [ ]:
# === 11c. 上传视频续接：把本地一段视频接到某镜成片末尾，再从其尾帧 AI 续写 ===
# 用法：VIDEO=你的视频路径(Drive/本地都行)，SCENE_NO=要接的镜号，COUNT=AI 续写段数(0=只拼接不续写)。
#   结果 = 该镜原成片 + 你的视频 + AI 续写。网页工作台每个分镜也有「上传视频续接」按钮做同样的事。
import requests, time, os
API='http://127.0.0.1:8000/api'; WS='/content/cael_video_out'
VIDEO = '/content/drive/MyDrive/my_clip.mp4'   # ← 改成你的视频路径
SCENE_NO = 1                                    # ← 接到第几镜
COUNT = 1                                       # AI 续写段数(0=只拼接)
pid = requests.get(f"{API}/pipeline/projects", params={"workspace": WS}).json()["projects"][0]["project_id"]
scs = requests.get(f"{API}/pipeline/project/{pid}", params={"workspace": WS}, timeout=90).json().get("scenes", [])
sid = next((s["scene_id"] for s in scs if s["scene_number"] == SCENE_NO), None)
if not sid:
    print(f"没找到镜号 {SCENE_NO}")
elif not os.path.exists(VIDEO):
    print(f"视频不存在: {VIDEO}")
else:
    with open(VIDEO, "rb") as f:
        r = requests.post(f"{API}/pipeline/upload_continue_video",
                          data={"scene_id": sid, "workspace": WS, "count": COUNT},
                          files={"file": (os.path.basename(VIDEO), f, "video/mp4")}, timeout=600)
    job = r.json().get("job_id"); print("任务:", job, "（拼接你的视频 + AI 续写中…）")
    t0 = time.time(); time.sleep(3)
    while True:
        act = requests.get(f"{API}/pipeline/jobs", params={"project_id": pid}, timeout=30).json().get("jobs", [])
        if not any(j["job_id"] == job for j in act):
            print(f"✅ 续接完成（{int(time.time() - t0)}s）"); break
        print(f"  续接中… ({int(time.time() - t0)}s)")
        if time.time() - t0 > 3600:
            print("超时，看 comfyui.log"); break
        time.sleep(15)
    sc = next((s for s in requests.get(f"{API}/pipeline/project/{pid}", params={"workspace": WS}, timeout=90).json().get("scenes", []) if s["scene_id"] == sid), {})
    v = sc.get("video"); print("该镜成片:", v.get("url") if isinstance(v, dict) else v)

In [ ]:
# === 实时日志（随时看出图/出片在画什么；服务已脱离内核，停本格只停 tail、不杀服务）===
# 看到 KSampler 的进度条 % = 正在采样出图/出片；卡住或 traceback 也会在这显形。
!tail -n 40 -f /content/comfyui.log

---
## 人物 LoRA 训练（FLUX-dev 系，含 Fluxed Up）

出图底模为 **FLUX-dev 系**（如无审查 Fluxed Up），训练用 **`is_flux:True`**（L3 已设；底模须与出图同源）。
虚构角色无照片 → 先用出图做一套同脸训练集（见 L2 注释）。**仅原创虚构成年角色，遵守合规前置。**

In [ ]:
# === L1. ai-toolkit(持久化 Drive，不重 clone) + pip缓存 + PuLID ===
import os, sys; sys.path.insert(0,'/content/mirage/colab'); import persist
CHAR='caelan'; TRIGGER=CHAR
TOOLS=globals().get('TOOLS','/content/drive/MyDrive/mirage_tools')
PIP_CACHE=globals().get('PIP_CACHE', TOOLS+'/pip_cache')
AITK_DRIVE=TOOLS+'/ai-toolkit'; AITK='/content/ai-toolkit'
WORK='/content/lora_work'; LORA_OUT='/content/ComfyUI/models/loras'; DS=WORK+'/'+CHAR+'_dataset'
for p in (TOOLS, PIP_CACHE, WORK, DS, LORA_OUT): os.makedirs(p, exist_ok=True)
if not os.path.isdir(AITK_DRIVE+'/.git'):
    !git clone https://github.com/ostris/ai-toolkit {AITK_DRIVE}
    !cd {AITK_DRIVE} && git submodule update --init --recursive
else:
    !cd {AITK_DRIVE} && git pull -q && git submodule update --init --recursive -q
persist.link_dir(AITK_DRIVE, AITK)                       # 软链回 /content（重连不重 clone）
!cd {AITK} && pip -q install --cache-dir {PIP_CACHE} -r requirements.txt
PW='/content/ComfyUI/models/pulid/pulid_flux_v0.9.1.safetensors'   # 在 models/pulid → 已软链 Drive
if not (os.path.exists(PW) and os.path.getsize(PW)>0):
    !wget -q -O {PW} https://huggingface.co/guozinan/PuLID/resolve/main/pulid_flux_v0.9.1.safetensors
print('ai-toolkit(Drive)+ pip缓存(Drive)+ PuLID 就绪; CHAR=', CHAR)

In [ ]:
# === L2. 数据集(16-20 张同一张脸)+ 打标 ===
# 训练集要「同一张脸的多角度图」。虚构角色获取法:
#  (a) 用工作台『一键出图』给该角色出多张 → 下载放进 DS,人工挑脸一致的 ≥5 张(越一致越好);
#  (b) 先出 1 张满意的种子脸,再以它做 img2img 换角度批量出(脸更稳)。
# 纯文生图每张脸都不同 → 直接拿来训会糊,务必挑/控同脸。
import os
BASE={'caelan':'caelan, teenage boy, messy black hair, sharp green eyes, fair skin',
      'roland':'roland, young noble man, slicked blonde hair, arrogant face',
      'seraphina':'seraphina, noblewoman, silver hair, haughty elegant face'}.get(CHAR, CHAR+', character')
pngs=[f for f in os.listdir(DS) if f.lower().endswith(('.png','.jpg','.jpeg'))]
for f in pngs:
    open(os.path.join(DS, os.path.splitext(f)[0]+'.txt'),'w',encoding='utf-8').write(
        BASE+', portrait, detailed face')
print(f'数据集 {len(pngs)} 张,已打标。建议 16-20 张同脸多角度/表情。')

In [ ]:
# === L3. 写训练配置(FLUX-dev 系，含 Fluxed Up; is_flux; A100-40G) ===
import yaml, os
# 训练底模与出图同源(FLUX-dev 系，如 Fluxed Up)。FLUX-dev 用 is_flux=True。
BASE_MODEL = os.environ.get('LORA_TRAIN_BASE', 'black-forest-labs/FLUX.1-dev')
cfg={'job':'extension','config':{'name':CHAR+'_lora','process':[{
  'type':'sd_trainer','training_folder':WORK+'/output','device':'cuda:0',
  'network':{'type':'lora','linear':32,'linear_alpha':16},
  'save':{'dtype':'float16','save_every':500,'max_step_saves_to_keep':2},
  'datasets':[{'folder_path':DS,'caption_ext':'txt','resolution':[512],'cache_latents_to_disk':True}],
  'train':{'batch_size':2,'steps':2000,'gradient_accumulation_steps':1,'train_unet':True,
           'train_text_encoder':False,'optimizer':'adamw8bit','lr':1e-4,'lr_scheduler':'cosine',
           'dtype':'bf16','gradient_checkpointing':True},
  'model':{'name_or_path':BASE_MODEL,'is_flux':True,'quantize':False},
  'sample':{'sample_every':500,
            'prompts':[TRIGGER+', portrait, studio lighting']},
}]}}
os.makedirs(WORK+'/output', exist_ok=True)
CFG=WORK+'/'+CHAR+'_train.yaml'; yaml.safe_dump(cfg, open(CFG,'w'))
print('训练配置(is_flux/FLUX-dev):', CFG)

In [ ]:
# === L4. 开训 + 产物落 loras(已软链 Drive) + 登记到项目 ===
import os, glob, shutil, sys
FINAL=LORA_OUT+'/'+CHAR+'_lora.safetensors'
if os.path.exists(FINAL):
    print('已训过,跳过:', FINAL)
else:
    !cd {AITK} && python run.py {CFG}
    outs=sorted(glob.glob(WORK+'/output/'+CHAR+'_lora/*.safetensors'))
    if outs: shutil.copy(outs[-1], FINAL); print('LoRA →', FINAL, '(在 loras/ → 已软链 Drive)')
    else: print('❌ 没产出 safetensors,看上面训练日志(多半是底模/ai-toolkit 版本)')
if os.path.exists(FINAL):
    sys.path.insert(0,'/content/mirage')
    from mirage.app.pipeline import runtime
    runtime.set_workspace('/content/cael_video_out')
    runtime.set_model_config(trigger_word=TRIGGER, flux_lora=CHAR+'_lora.safetensors')
    print('已登记 trigger=%s lora=%s_lora.safetensors → 出图自动加载' % (TRIGGER, CHAR))